In [85]:
import pandas as pd
import numpy as np

In [86]:
# Dataset loaded
df = pd.read_csv("D:/coding programs/ML Projects/churnsense-ai/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [87]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [88]:
# Finding missing values
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [89]:
# Removing unnecessary columns which are not needed for ml model
df.drop(['customerID'], axis=1, inplace=True)

In [90]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [91]:
# Unique variable exists in the columns
df['gender'].unique()

<StringArray>
['Female', 'Male']
Length: 2, dtype: str

In [92]:
# Convert TotalCharges to int dtype
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

In [93]:
df['TotalCharges'].isna().sum()

np.int64(11)

In [94]:
# Separate features and target
X = df.drop('Churn', axis = 1)
y = df['Churn']

In [95]:
# Encoding the target
y = y.map({
    'No': 0,
    'Yes': 1
})

In [96]:
y.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [97]:
# Split training and testing data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42, stratify=y)

In [98]:
# Temporarily fill na values in TotalCharges to median of the training Dataset
# Learn median only from training data
total_charges_median = X_train['TotalCharges'].median()

df['TotalCharges'] = df['TotalCharges'].fillna(
    df['TotalCharges'].median()
)

# Fill training data
X_train['TotalCharges'] = X_train['TotalCharges'].fillna(
    total_charges_median
)

# Fill test data using the training median
X_test['TotalCharges'] = X_test['TotalCharges'].fillna(
    total_charges_median
)

In [99]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5634, 19)
X_test: (1409, 19)
y_train: (5634,)
y_test: (1409,)


In [100]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

In [101]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [102]:
# Identify categorical feature columns
categorical_columns = X_train.select_dtypes(
    include=['object', 'str']
).columns

In [103]:
# Identify numerical feature columns
numerical_columns = X_train.select_dtypes(
    include=['float64', 'int64']
).columns

In [104]:
for col in categorical_columns:
    print(col)
    print(list(df[col].unique()))

gender
['Female', 'Male']
Partner
['Yes', 'No']
Dependents
['No', 'Yes']
PhoneService
['No', 'Yes']
MultipleLines
['No phone service', 'No', 'Yes']
InternetService
['DSL', 'Fiber optic', 'No']
OnlineSecurity
['No', 'Yes', 'No internet service']
OnlineBackup
['Yes', 'No', 'No internet service']
DeviceProtection
['No', 'Yes', 'No internet service']
TechSupport
['No', 'Yes', 'No internet service']
StreamingTV
['No', 'Yes', 'No internet service']
StreamingMovies
['No', 'Yes', 'No internet service']
Contract
['Month-to-month', 'One year', 'Two year']
PaperlessBilling
['Yes', 'No']
PaymentMethod
['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']


In [105]:
ordinal_column = ['Contract']

onehot_column = [
    column 
    for column in categorical_columns 
    if not column in ordinal_column
]

In [106]:
onehot_column

['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'PaperlessBilling',
 'PaymentMethod']

In [107]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'ordinal',
            OrdinalEncoder(
                categories=[
                    ['Month-to-month', 'One year', 'Two year']
                ],
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            ordinal_column
        ),

        (
            'onehot',
            OneHotEncoder(
                drop="if_binary",
                categories='auto',
                handle_unknown='ignore',
                sparse_output=False
            ),
            onehot_column
        )
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

In [108]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_train_transformed.shape

(5634, 38)

In [109]:
preprocessor.named_transformers_['ordinal'].categories_

[array(['Month-to-month', 'One year', 'Two year'], dtype=object)]

In [110]:
X_test_transformed = preprocessor.transform(X_test)
X_test_transformed.shape


(1409, 38)

In [111]:
feature_names = preprocessor.get_feature_names_out()
feature_names

array(['Contract', 'gender_Male', 'Partner_Yes', 'Dependents_Yes',
       'PhoneService_Yes', 'MultipleLines_No',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_DSL', 'InternetService_Fiber optic',
       'InternetService_No', 'OnlineSecurity_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No', 'OnlineBackup_No internet service',
       'OnlineBackup_Yes', 'DeviceProtection_No',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No', 'TechSupport_No internet service',
       'TechSupport_Yes', 'StreamingTV_No',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No', 'StreamingMovies_No internet service',
       'StreamingMovies_Yes', 'PaperlessBilling_Yes',
       'PaymentMethod_Bank transfer (automatic)',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       

In [112]:
X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=X_train.index
)

X_test_transformed = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
    index=X_test.index
)

In [113]:
X_train_transformed.head()

,Contract,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,InternetService_Fiber optic,...,StreamingMovies_Yes,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
3738,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,35.0,49.20,1701.65
3151,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,15.0,75.10,1151.55
4860,2.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,13.0,40.55,590.35
3867,2.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,26.0,73.50,1905.70
3810,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,44.55,44.55


In [114]:
X_train_transformed.shape

(5634, 38)

In [115]:
X_train_transformed.columns

Index(['Contract', 'gender_Male', 'Partner_Yes', 'Dependents_Yes',
       'PhoneService_Yes', 'MultipleLines_No',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_DSL', 'InternetService_Fiber optic',
       'InternetService_No', 'OnlineSecurity_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No', 'OnlineBackup_No internet service',
       'OnlineBackup_Yes', 'DeviceProtection_No',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No', 'StreamingMovies_No internet service',
       'StreamingMovies_Yes', 'PaperlessBilling_Yes',
       'PaymentMethod_Bank transfer (automatic)',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'Senior

In [116]:
y_train.isna().sum()

np.int64(0)

In [117]:
type(X_train_transformed)

pandas.DataFrame

In [118]:
X_train_transformed = X_train_transformed.rename(
    columns={
        "gender_Male": "gender",
        "Partner_Yes": "Partner",
        "Dependents_Yes": "Dependents",
        "PhoneService_Yes": "PhoneService",
        "PaperlessBilling_Yes": "PaperlessBilling"
    }
)
X_test_transformed = X_test_transformed.rename(
    columns={
        "gender_Male": "gender",
        "Partner_Yes": "Partner",
        "Dependents_Yes": "Dependents",
        "PhoneService_Yes": "PhoneService",
        "PaperlessBilling_Yes": "PaperlessBilling"
    }
)

In [119]:
train_processed = pd.concat(
    [
        X_train_transformed.reset_index(drop=True),
        y_train.reset_index(drop=True)
    ],
    axis=1
)

In [120]:
train_processed.isna().sum()

Contract                                   0
gender                                     0
Partner                                    0
Dependents                                 0
PhoneService                               0
MultipleLines_No                           0
MultipleLines_No phone service             0
MultipleLines_Yes                          0
InternetService_DSL                        0
InternetService_Fiber optic                0
InternetService_No                         0
OnlineSecurity_No                          0
OnlineSecurity_No internet service         0
OnlineSecurity_Yes                         0
OnlineBackup_No                            0
OnlineBackup_No internet service           0
OnlineBackup_Yes                           0
DeviceProtection_No                        0
DeviceProtection_No internet service       0
DeviceProtection_Yes                       0
TechSupport_No                             0
TechSupport_No internet service            0
TechSuppor

In [121]:
X_test.shape

(1409, 19)

In [122]:
test_processed = pd.concat(
    [
        X_test_transformed.reset_index(drop=True),
        y_test.reset_index(drop=True)
    ],
    axis=1
)

In [123]:
train_processed.columns

Index(['Contract', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines_No', 'MultipleLines_No phone service',
       'MultipleLines_Yes', 'InternetService_DSL',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No', 'OnlineSecurity_No internet service',
       'OnlineSecurity_Yes', 'OnlineBackup_No',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No', 'DeviceProtection_No internet service',
       'DeviceProtection_Yes', 'TechSupport_No',
       'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No', 'StreamingMovies_No internet service',
       'StreamingMovies_Yes', 'PaperlessBilling',
       'PaymentMethod_Bank transfer (automatic)',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'SeniorCitizen', 'tenure', '

In [124]:
# Numerical columns that should remain float
numerical_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

# Target column
target_col = "Churn"

encoded_cols = [
    col for col in train_processed.columns
    if col not in numerical_cols + [target_col]
]

train_processed[encoded_cols] = train_processed[encoded_cols].astype("int64")
test_processed[encoded_cols] = test_processed[encoded_cols].astype("int64")



In [125]:
# Save both DataFrames to a separate csv file
train_processed.to_csv(
    "../data/processed/train_processed.csv",
    index=False
)

test_processed.to_csv(
    "../data/processed/test_processed.csv",
    index=False
)


In [126]:
train_processed.head()

,Contract,gender,Partner,Dependents,PhoneService,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,InternetService_Fiber optic,...,PaperlessBilling,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn
0,0,1,0,0,0,0,1,0,1,0,...,0,0,0,1,0,0,35.0,49.20,1701.65,0
1,0,1,1,1,1,1,0,0,0,1,...,0,0,0,0,1,0,15.0,75.10,1151.55,0
2,2,1,1,1,0,0,1,0,1,0,...,0,0,0,0,1,0,13.0,40.55,590.35,0
3,2,0,1,0,1,1,0,0,1,0,...,1,0,1,0,0,0,26.0,73.50,1905.70,0
4,0,1,1,1,1,1,0,0,1,0,...,0,0,0,1,0,0,1.0,44.55,44.55,0
